# 03 – Name score cut-offs

Input to decisions D7 and D8: where "exact", "strong" and "weak" name outcomes start.
Scores come from `data/output/phase_a_features.csv` (run `python run.py` first).

`name_sort` is `token_sort_ratio` on core names (legal form, country word and branch removed); `name_set` is the looser `token_set_ratio`.
For each input, the best candidate is the highest `name_sort` among candidates not in a different country.

In [1]:
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 45)

f = pd.read_csv("../data/output/phase_a_features.csv")
BANDS = [-1, 49.99, 69.99, 79.99, 84.99, 89.99, 94.99, 99.99, 100]
LABELS = ["<50", "50-69", "70-79", "80-84", "85-89", "90-94", "95-99", "100"]

ok = f[f.country != "different"]
best = ok.sort_values(["name_sort", "name_set"], ascending=False).drop_duplicates("input_row_key").copy()
best["band"] = pd.cut(best.name_sort, bins=BANDS, labels=LABELS)
print(f"{best.input_row_key.nunique()} of {f.input_row_key.nunique()} inputs have a candidate not in a different country")
best.band.value_counts().reindex(LABELS[::-1]).rename("inputs").to_frame()

579 of 592 inputs have a candidate not in a different country


,inputs
band,
100,393
95-99,0
90-94,6
85-89,10
80-84,15
70-79,45
50-69,77
<50,33


## 1. Every input with a best score from 80 to 99

This is where the "strong" cut-off decides the outcome. `city` and `street` are shown because a strong name alone never auto-matches (D4).

In [2]:
COLS = ["input_row_key", "input_core", "cand_core", "name_sort", "name_set", "country", "city", "street"]
best[best.name_sort.between(80, 99.99)].sort_values("name_sort", ascending=False)[COLS]

,input_row_key,input_core,cand_core,name_sort,name_set,country,city,street
2207,498,TATA CONSULTANCY SERVICES,TATA CONSULTANCY SERVICES SEP,92.592593,100.000000,unknown,unknown,unknown
2447,540,TP LINK INTERNATIONAL,LINK INTERNATIONAL,92.307692,100.000000,same,different,different
1623,391,PAKISTAN INSTITUTE OF NUCLEAR SCIENCE AND...,PAKISTAN INSTITUTE OF NUCLEAR SCIENCE AND...,92.035398,100.000000,same,same,unknown
468,182,EWORK GROUP,E WORKGROUP,90.909091,90.909091,unknown,unknown,unknown
2950,99,COMBA TECHNOLOGIES,OM TECHNOLOGIES,90.909091,90.909091,same,different,different
191,132,DELOITTE CONSULTING SEA,DELOITTE CONSULTING,90.476190,100.000000,same,same,street_only
892,259,INDEPENDENT MEDIA,INDEPENDENT PAK MEDIA,89.473684,100.000000,same,different,unknown
495,188,FACEBOOK,FACEBOOK 3,88.888889,100.000000,same,different,unknown
2214,499,TCT MOBILE INTERNATIONAL,MOBILE TECH INTERNATIONAL HK,88.461538,90.909091,same,different,unknown
1532,375,MEDIA DIRECTION,NEW MEDIA DIRECTION,88.235294,100.000000,same,same,different


## 2. Samples from lower bands

In [3]:
for band in ["70-79", "50-69", "<50"]:
    print(f"== {band}")
    display(best[best.band == band].sample(8, random_state=7)[COLS])

== 70-79


,input_row_key,input_core,cand_core,name_sort,name_set,country,city,street
2863,83,C AND W SERVICES,CW AERO SERVICES,75.000000,81.250000,same,same,unknown
922,264,INLUMI,INLUMI DOT,75.000000,100.000000,same,different,different
2087,476,SQUARETRADE EUROPE,SQUARETRADE,75.862069,100.000000,unknown,unknown,unknown
1410,352,NEWS PUBLICATIONS,JUICY PUBLICATIONS,74.285714,82.758621,same,same,different
2631,574,WOODLANDS TRANSPORT SOLUTIONS,WOODLANDS TRANSPORT,79.166667,100.000000,same,same,different
1990,458,SINGTEL GLOBAL INVESTMENT,ROBO GLOBAL INVESTMENT,76.595745,87.179487,same,same,unknown
91,114,CROWDSTRIKE SERVICES,CROWDSTRIKE SAUDI,75.675676,78.571429,same,different,different
575,201,GEO ENTERTAINMENT TELEVISION,GEO ENTERTAINMENT,75.555556,100.000000,same,unknown,unknown


== 50-69


,input_row_key,input_core,cand_core,name_sort,name_set,country,city,street
625,210,GOOGLE,GOOGLE COMMERCE,57.142857,100.000000,same,same,street_only
2887,88,CENTERFORENINGEN FISKETORVET SHOPPING CEN...,CENTERFORENINGEN FOR NYVANGSPARKEN,60.240964,64.000000,same,unknown,unknown
513,190,FASTBRIDGE,FAST BRIDGE,57.142857,57.142857,unknown,unknown,unknown
1639,394,VARSTA SOFTWARE,VARSTA PETRI,66.666667,66.666667,same,different,different
202,134,DELTA EJENDOM AALBORG,AALBORG ENGINEERING,60.000000,60.000000,same,same,different
1348,340,NAGARRO ENTERPRISE SERVICES,TECHNOWELL ENTERPRISE SERVICES,66.666667,82.608696,same,different,unknown
1224,318,MARSH AND MCLENNAN AGENCIES,GULBRANDSEN AGENCIES,63.829787,63.829787,same,same,unknown
1587,385,ORACLE SVENSKA,ORACLE PET,66.666667,75.000000,same,unknown,unknown


== <50


,input_row_key,input_core,cand_core,name_sort,name_set,country,city,street
859,252,HYDRO ENERGI,HYDRO REIN,45.454545,72.727273,same,same,exact
333,158,EDOTCO TOWERS,WORLDWIDE PHARMA CONSULTANT IN,27.906977,27.906977,same,unknown,unknown
1891,440,SAS INSTITUTE,SAS CTRL,47.619048,54.545455,same,different,different
1963,453,SIAE MICROELETTRONICA,CUBE COMMUNICATION,41.025641,41.025641,same,different,unknown
882,257,IMI MOBILE VAS,BANGLA TRAC MIAKI VAS,40.000000,40.000000,same,unknown,unknown
2123,482,SUBEX ASIA PACIFIC,SUBEX,43.478261,100.000000,unknown,unknown,unknown
2528,555,UR CONSULT,ROESTED,35.294118,35.294118,same,same,street_only
541,196,SYKES ENTERPRISES,DALEN LEGESENTER,36.363636,36.363636,same,same,street_only


## 3. Subset matches (D8)

Best candidates whose words are all contained in the input name or the other way round (`name_set` = 100, `name_sort` < 100).

In [4]:
subset = best[(best.name_set == 100) & (best.name_sort < 100)]
print(f"{len(subset)} inputs have a subset match as their best candidate")
subset.sample(15, random_state=7)[COLS]

58 inputs have a subset match as their best candidate


,input_row_key,input_core,cand_core,name_sort,name_set,country,city,street
1861,435,SALLING GROUP FORSIKRING,SALLING GROUP,70.270270,100.0,same,different,unknown
13,10,ACCENTURE SERVICES,ACCENTURE,66.666667,100.0,same,different,different
485,186,EXTENDA RETAIL AS AVD OSLO,EXTENDA RETAIL,70.000000,100.0,same,same,exact
955,270,IPG MEDIABRANDS,IPG MEDIABRANDS APAC,85.714286,100.0,same,unknown,unknown
2315,516,TELENOR GLOBAL SHARED SERVICES,SHARED SERVICES,66.666667,100.0,same,different,unknown
2631,574,WOODLANDS TRANSPORT SOLUTIONS,WOODLANDS TRANSPORT,79.166667,100.0,same,same,different
1340,34,AMERICAN EXPRESS EUROPE,AMERICAN EXPRESS,82.051282,100.0,unknown,unknown,unknown
38,104,COMMSCOPE TECHNOLOGIES,COMMSCOPE,58.064516,100.0,same,unknown,unknown
837,249,HUBER SUHNER ELECTRONICS,HUBER SUHNER,66.666667,100.0,unknown,unknown,unknown
1260,325,MEISEL EIENDOM,MEISEL,60.000000,100.0,same,unknown,unknown


## 4. Ties at a perfect name score

Inputs where two or more candidates (not in a different country) score 100. Location layers and the fail-safe check must separate them.

In [5]:
ties = ok[ok.name_sort == 100].groupby("input_row_key").size()
print(f"{(ties >= 2).sum()} inputs have 2+ candidates at name_sort = 100")

119 inputs have 2+ candidates at name_sort = 100
